# 04 Infer Cluster ADS
Bestes Modell laden, ADS blockweise scoren, DBSCAN-Clustering, Exporte.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json
DEVICE = "auto"
import json
import numpy as np
import pandas as pd


ROOT: /home/ubuntu/Author_Name_Disambiguation


In [2]:
from src.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from src.common.io_schema import read_parquet
from src.approaches.nand.infer_pairs import score_pairs_with_checkpoint
from src.approaches.nand.cluster import cluster_blockwise_dbscan, resolve_dbscan_eps
from src.approaches.nand.export import build_publication_author_mapping

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
CLUSTER_CFG = CTX["CLUSTER"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "checkpoints", "pair_scores", "clusters", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "04_run_consistency.json",
    extras={"notebook": "04_infer_cluster_ads", "latest_context_path": str(latest_context_path)},
)

subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
pair_score_dir = RUN_DIRS["pair_scores"]
cluster_dir = RUN_DIRS["clusters"]
metrics_dir = RUN_DIRS["metrics"]

ads_mentions = read_parquet(subset_dir / f"ads_mentions_{RUN_STAGE}.parquet")
ads_pairs = read_parquet(subset_dir / f"ads_pairs_{RUN_STAGE}.parquet")
ads_chars = np.load(emb_dir / f"ads_chars2vec_{RUN_STAGE}.npy")
ads_text = np.load(emb_dir / f"ads_specter_{RUN_STAGE}.npy")

with (metrics_dir / "03_train_manifest.json").open("r", encoding="utf-8") as f:
    train_manifest = json.load(f)

best_checkpoint = train_manifest["best_checkpoint"]
best_threshold = float(train_manifest["best_threshold"])

cluster_cfg_used = json.loads(json.dumps(CLUSTER_CFG))
resolved_eps, eps_meta = resolve_dbscan_eps(cluster_cfg_used, cosine_threshold=best_threshold)
cluster_cfg_used["eps"] = resolved_eps

clustering_config_used_path = metrics_dir / "04_clustering_config_used.json"
with clustering_config_used_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "run_id": RUN_ID,
            "run_stage": RUN_STAGE,
            "best_threshold": best_threshold,
            "eps_resolution": eps_meta,
            "cluster_config_used": cluster_cfg_used,
        },
        f,
        indent=2,
    )

print("RUN_ID:", RUN_ID)
print("Best checkpoint:", best_checkpoint)
print("Best threshold:", best_threshold)
print("Resolved DBSCAN eps:", resolved_eps)
print("Clustering config used:", clustering_config_used_path)


RUN_ID: smoke_20260213T103348Z_d2c63f9f
Best checkpoint: /home/ubuntu/Author_Name_Disambiguation/artifacts/checkpoints/smoke_20260213T103348Z_d2c63f9f/smoke_20260213T103348Z_d2c63f9f_seed1.pt
Best threshold: -0.126
Resolved DBSCAN eps: 0.85
Clustering config used: /home/ubuntu/Author_Name_Disambiguation/artifacts/metrics/smoke_20260213T103348Z_d2c63f9f/04_clustering_config_used.json


In [3]:
pair_scores_path = pair_score_dir / f"ads_pair_scores_{RUN_STAGE}.parquet"
pair_scores = score_pairs_with_checkpoint(
    mentions=ads_mentions,
    pairs=ads_pairs,
    chars2vec=ads_chars,
    text_emb=ads_text,
    checkpoint_path=best_checkpoint,
    output_path=pair_scores_path,
    device=DEVICE,
)
print("Pair scores:", len(pair_scores), "->", pair_scores_path)


Pair scores: 500 -> /home/ubuntu/Author_Name_Disambiguation/artifacts/pair_scores/smoke_20260213T103348Z_d2c63f9f/ads_pair_scores_smoke.parquet


In [4]:
clusters_path = cluster_dir / f"ads_clusters_{RUN_STAGE}.parquet"
clusters = cluster_blockwise_dbscan(
    mentions=ads_mentions,
    pair_scores=pair_scores,
    cluster_config=cluster_cfg_used,
    output_path=clusters_path,
)
print("Clusters:", len(clusters), "->", clusters_path)


Clusters: 1000 -> /home/ubuntu/Author_Name_Disambiguation/artifacts/clusters/smoke_20260213T103348Z_d2c63f9f/ads_clusters_smoke.parquet


In [5]:
merged = ads_mentions[["mention_id", "block_key"]].merge(clusters, on=["mention_id", "block_key"], how="left")
cluster_size = merged.groupby(["block_key", "author_uid"]).size().rename("size").reset_index()

singleton_ratio = float((cluster_size["size"] == 1).mean()) if len(cluster_size) else 0.0
print("Singleton ratio:", singleton_ratio)
print("Cluster size describe:")
print(cluster_size["size"].describe())
display(cluster_size.groupby("block_key")["size"].agg(["count", "max"]).sort_values(["count", "max"], ascending=False).head(20))

diag = pair_scores.merge(
    clusters[["mention_id", "author_uid"]].rename(columns={"mention_id": "mention_id_1", "author_uid": "author_uid_1"}),
    on="mention_id_1",
    how="left",
).merge(
    clusters[["mention_id", "author_uid"]].rename(columns={"mention_id": "mention_id_2", "author_uid": "author_uid_2"}),
    on="mention_id_2",
    how="left",
)
diag["same_cluster"] = diag["author_uid_1"] == diag["author_uid_2"]

merged_low_conf = diag[(diag["same_cluster"]) & (diag["cosine_sim"] < best_threshold)].sort_values("cosine_sim").head(20)
split_high_sim = diag[(~diag["same_cluster"]) & (diag["cosine_sim"] >= best_threshold)].sort_values("cosine_sim", ascending=False).head(20)

print("Merged despite low confidence (top 20):")
display(merged_low_conf[["pair_id", "block_key", "cosine_sim", "distance", "author_uid_1", "author_uid_2"]])
print("Split despite high similarity (top 20):")
display(split_high_sim[["pair_id", "block_key", "cosine_sim", "distance", "author_uid_1", "author_uid_2"]])

with (metrics_dir / "04_cluster_qc.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "singleton_ratio": singleton_ratio,
            "merged_low_conf_count": int(len(merged_low_conf)),
            "split_high_sim_count": int(len(split_high_sim)),
            "cluster_count": int(len(cluster_size)),
        },
        f,
        indent=2,
    )


Singleton ratio: 0.41017488076311603
Cluster size describe:
count    629.000000
mean       1.589825
std        0.492257
min        1.000000
25%        1.000000
50%        2.000000
75%        2.000000
max        2.000000
Name: size, dtype: float64


,count,max
block_key,,
a.cameron,2,1
a.dalgarno,2,1
a.martin,2,1
a.thompson,2,1
a.treves,2,1
a.watson,2,1
a.wilson,2,1
a.wright,2,1
b.carter,2,1


Merged despite low confidence (top 20):


,pair_id,block_key,cosine_sim,distance,author_uid_1,author_uid_2


Split despite high similarity (top 20):


,pair_id,block_key,cosine_sim,distance,author_uid_1,author_uid_2
3,1976ApJ...203..132B::1__1977PhRvA..15..154L::2,a.dalgarno,0.147520,0.852480,a.dalgarno::0,a.dalgarno::1
373,1974A&A....30..241W::1__1992AJ....104..704S::4,r.ekers,0.143501,0.856499,r.ekers::0,r.ekers::1
259,1981ApJ...246.1014Y::0__1986ApJ...302..680Y::0,j.young,0.137519,0.862481,j.young::0,j.young::1
282,1975ApJ...195..715M::1__1985ApJS...59..447H::1,l.cowie,0.130114,0.869886,l.cowie::0,l.cowie::1
326,1987LNP...291..192S::0__1990AdSpR..10l.233M::6,m.smith,0.129521,0.870479,m.smith::0,m.smith::1
23,1975PhRvL..34..507T::0__1990PhLB..245..289D::161,a.thompson,0.127509,0.872491,a.thompson::0,a.thompson::1
242,1971ICRC....8..324S::0__1978JFM....88..223B::1,j.simpson,0.127470,0.872530,j.simpson::0,j.simpson::1
437,1967PhRv..162Q1750B::2__1992ZPhyC..56..509A::27,s.gupta,0.124263,0.875737,s.gupta::0,s.gupta::1
353,1974ApJ...190L.109H::4__1979BAAS...11R.782S::5,p.serlemitsos,0.123144,0.876856,p.serlemitsos::0,p.serlemitsos::1
291,1973NPhS..242..117P::5__1986ApJ...304..637M::0,l.maraschi,0.119795,0.880205,l.maraschi::0,l.maraschi::1


In [6]:
mention_export = cluster_dir / f"mention_author_uid_{RUN_STAGE}.parquet"
pub_export = cluster_dir / f"publication_authors_{RUN_STAGE}.parquet"

clusters.to_parquet(mention_export, index=False)
pub_map = build_publication_author_mapping(mentions=ads_mentions, clusters=clusters, output_path=pub_export)

print("mention export:", mention_export)
print("publication export:", pub_export)
print("publication rows:", len(pub_map))


mention export: /home/ubuntu/Author_Name_Disambiguation/artifacts/clusters/smoke_20260213T103348Z_d2c63f9f/mention_author_uid_smoke.parquet
publication export: /home/ubuntu/Author_Name_Disambiguation/artifacts/clusters/smoke_20260213T103348Z_d2c63f9f/publication_authors_smoke.parquet
publication rows: 1000
